### Libraries

In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import sys
import yaml
from data_loader import DataLoader

### Load data

In [13]:
config_file = os.path.join('..', 'config', 'config.yaml')
loader = DataLoader(config_file)
df = loader.load_raw_data()

Data loaded successfully: (32581, 12)


### Step 1: Define split ratios

In [14]:
TRAIN_SIZE = 0.70
VAL_SIZE   = 0.15
TEST_SIZE  = 0.15

print(f"Train: {TRAIN_SIZE*100}%")
print(f"Val:   {VAL_SIZE*100}%")
print(f"Test:  {TEST_SIZE*100}%")

Train: 70.0%
Val:   15.0%
Test:  15.0%


### Step 2: Set Random Seeds

In [15]:
RANDOM_STATE = 42
print(f"Random State: {RANDOM_STATE}")


Random State: 42


###  Step 3: Handle Missing Values in Target Columns

In [16]:
print("The shape of dataset before handling missing values:", df.shape)

The shape of dataset before handling missing values: (32581, 12)


In [17]:
print("Total Missing Value in Loan Status Column:",df["loan_status"].isna().sum())
print("Total Missing Value in Loan int rate Column:",df["loan_int_rate"].isna().sum())

Total Missing Value in Loan Status Column: 0
Total Missing Value in Loan int rate Column: 3116


In [22]:
df_classification = df.copy()
df_regression = df.copy()

In [23]:
df_regression = df_regression.dropna(subset=["loan_status", "loan_int_rate"])
print("The shape of dataset after handling missing values:", df_regression.shape)
print("Total Missing Value in Loan Status Column:",df_regression["loan_status"].isna().sum())
print("Total Missing Value in Loan int rate Column:",df_regression["loan_int_rate"].isna().sum())

The shape of dataset after handling missing values: (29465, 12)
Total Missing Value in Loan Status Column: 0
Total Missing Value in Loan int rate Column: 0


In [24]:
print(f"Classification dataset shape: {df_classification.shape}")
print(f"Regression dataset shape: {df_regression.shape}")

Classification dataset shape: (32581, 12)
Regression dataset shape: (29465, 12)


###  Step 4: Separate Features (X) and Targets (y)

In [26]:
X_cls = df_classification.drop(columns=["loan_status"])
y_cls = df_classification["loan_status"]

In [27]:
X_reg = df_regression.drop(columns=["loan_int_rate"])
y_reg = df_regression["loan_int_rate"]

In [28]:
print(f"Classification features shape: {X_cls.shape}")
print(f"Classification target shape: {y_cls.shape}")
print(f"Regression features shape: {X_reg.shape}")
print(f"Regression target shape: {y_reg.shape}")

Classification features shape: (32581, 11)
Classification target shape: (32581,)
Regression features shape: (29465, 11)
Regression target shape: (29465,)


###  Step 5: Stratified Split for Classification
Stratified = preserves class distribution

In [30]:
X_train_cls, X_temp_cls, y_train_cls, y_temp_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.15,
    random_state=42,
    stratify=y_cls
)

In [31]:
X_val_cls, X_test_cls, y_val_cls, y_test_cls = train_test_split(
    X_temp_cls,
    y_temp_cls,
    test_size=0.5,   # 50% of 15% = 7.5%
    random_state=42,
    stratify=y_temp_cls
)

In [32]:
print("Classification Splits:")
print("X_train_cls:", X_train_cls.shape)
print("X_val_cls:  ", X_val_cls.shape)
print("X_test_cls: ", X_test_cls.shape)

Classification Splits:
X_train_cls: (27693, 11)
X_val_cls:   (2444, 11)
X_test_cls:  (2444, 11)


### Step 6: Regular Split for Regression

In [33]:
X_train_reg, X_temp_reg, y_train_reg, y_temp_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.15,
    random_state=42,
)

In [34]:
X_val_reg, X_test_reg, y_val_reg, y_test_reg = train_test_split(
    X_temp_reg,
    y_temp_reg,
    test_size=0.5,   # 50% of 15% = 7.5%
    random_state=42,
)

In [35]:
print("Regression Splits:")
print("X_train_reg:", X_train_reg.shape)
print("X_val_reg:  ", X_val_reg.shape)
print("X_test_reg: ", X_test_reg.shape)

Regression Splits:
X_train_reg: (25045, 11)
X_val_reg:   (2210, 11)
X_test_reg:  (2210, 11)


### Step 7: Clustering Uses Train Set Only

In [36]:
# CLUSTERING
# Clustering is unsupervised — no target column
# It uses training features only
# X_train_cls will be used for clustering
# No separate split needed for clustering
print("Clustering will use X_train_cls for training")
print(f"Clustering dataset shape: {X_train_cls.shape}")

Clustering will use X_train_cls for training
Clustering dataset shape: (27693, 11)


###  Step 8: Verify Split Sizes

In [37]:
print("Classification Report:")
print("X_train_cls:", X_train_cls.shape , "Percentage:", X_train_cls.shape[0]/X_cls.shape[0]*100)
print("X_val_cls:  ", X_val_cls.shape , "Percentage:", X_val_cls.shape[0]/X_cls.shape[0]*100)
print("X_test_cls: ", X_test_cls.shape , "Percentage:", X_test_cls.shape[0]/X_cls.shape[0]*100)
print("Total:", len(df_classification), "Percentage:", (len(df_classification))/X_cls.shape[0]*100)

Classification Report:
X_train_cls: (27693, 11) Percentage: 84.99739111752248
X_val_cls:   (2444, 11) Percentage: 7.501304441238759
X_test_cls:  (2444, 11) Percentage: 7.501304441238759
Total: 32581 Percentage: 100.0


In [38]:
print("Regression Report:")
print("X_train_reg:", X_train_reg.shape , "Percentage:", X_train_reg.shape[0]/X_reg.shape[0]*100)
print("X_val_reg:  ", X_val_reg.shape , "Percentage:", X_val_reg.shape[0]/X_reg.shape[0]*100)
print("X_test_reg: ", X_test_reg.shape , "Percentage:", X_test_reg.shape[0]/X_reg.shape[0]*100)
print("Total:", len(df_regression), "Percentage:", (len(df_regression))/X_reg.shape[0]*100)

Regression Report:
X_train_reg: (25045, 11) Percentage: 84.99915153572034
X_val_reg:   (2210, 11) Percentage: 7.500424232139827
X_test_reg:  (2210, 11) Percentage: 7.500424232139827
Total: 29465 Percentage: 100.0


### Step 9: Verify Class Distribution Preserved

In [60]:
print("\t\t   Class 1\t" , "        Class 0")
print("Original Dataset: ", 100 - df_classification['loan_status'].value_counts(normalize=True).iloc[0]*100, "%\t", df_classification['loan_status'].value_counts(normalize=True).iloc[0]*100, "%")
print("Train Dataset   : ", 100 - y_train_cls.value_counts(normalize=True).iloc[0]*100, "%\t", y_train_cls.value_counts(normalize=True).iloc[0]*100, "%")
print("Val Dataset     : ", 100 - y_val_cls.value_counts(normalize=True).iloc[0]*100, "%\t", y_val_cls.value_counts(normalize=True).iloc[0]*100, "%")
print("Test Dataset    : ", 100 - y_test_cls.value_counts(normalize=True).iloc[0]*100, "%\t", y_test_cls.value_counts(normalize=True).iloc[0]*100, "%")



		   Class 1	         Class 0
Original Dataset:  21.81639605905282 %	 78.18360394094718 %
Train Dataset   :  21.81778788863612 %	 78.18221211136388 %
Val Dataset     :  21.808510638297875 %	 78.19148936170212 %
Test Dataset    :  21.808510638297875 %	 78.19148936170212 %


### Step 10: Check No Overlap Between Splits

In [61]:
# Get indices
train_idx = set(X_train_cls.index)
val_idx = set(X_val_cls.index)
test_idx = set(X_test_cls.index)

# Check overlap
overlap = train_idx.intersection(val_idx)
overlap_test = train_idx.intersection(test_idx)
overlap_val = val_idx.intersection(test_idx)

print(f"Overlap between Train and Val: {len(overlap)} samples")
print(f"Overlap between Train and Test: {len(overlap_test)} samples")
print(f"Overlap between Val and Test: {len(overlap_val)} samples")

Overlap between Train and Val: 0 samples
Overlap between Train and Test: 0 samples
Overlap between Val and Test: 0 samples


###  Step 11: Save All Splits

In [67]:
# Load config
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

    root_dir = os.path.dirname(os.path.dirname(os.path.abspath(config_file)))
    splits_path = os.path.join(root_dir, config['paths']['splits_data'])

# Create folder if it doesn't exist
os.makedirs(splits_path, exist_ok=True)

In [68]:
# Helper function to save splits
def save_split(X, y, prefix, task_type):
    X.to_csv(os.path.join(splits_path, f"X_{prefix}_{task_type}.csv"), index=False)
    y.to_csv(os.path.join(splits_path, f"y_{prefix}_{task_type}.csv"), index=False)


In [69]:
# Save Classification Splits
save_split(X_train_cls, y_train_cls, "train", "cls")
save_split(X_val_cls,   y_val_cls,   "val",   "cls")
save_split(X_test_cls,  y_test_cls,  "test",  "cls")

In [70]:
# Save Regression Splits
save_split(X_train_reg, y_train_reg, "train", "reg")
save_split(X_val_reg,   y_val_reg,   "val",   "reg")
save_split(X_test_reg,  y_test_reg,  "test",  "reg")

### Step 12: Verify Saved Splits

In [73]:
X_train_cls = pd.read_csv(os.path.join(splits_path, "X_train_cls.csv"))
y_train_cls = pd.read_csv(os.path.join(splits_path, "y_train_cls.csv"))
X_test_cls = pd.read_csv(os.path.join(splits_path, "X_test_cls.csv"))
y_test_cls = pd.read_csv(os.path.join(splits_path, "y_test_cls.csv"))
X_val_cls = pd.read_csv(os.path.join(splits_path, "X_val_cls.csv"))
y_val_cls = pd.read_csv(os.path.join(splits_path, "y_val_cls.csv"))

print("""
      X_train_cls shape: {}, y_train_cls shape: {}
      X_val_cls shape: {}, y_val_cls shape: {}
      X_test_cls shape: {}, y_test_cls shape: {}
      """.format(X_train_cls.shape, y_train_cls.shape, X_val_cls.shape, y_val_cls.shape, X_test_cls.shape, y_test_cls.shape))


      X_train_cls shape: (27693, 11), y_train_cls shape: (27693, 1)
      X_val_cls shape: (2444, 11), y_val_cls shape: (2444, 1)
      X_test_cls shape: (2444, 11), y_test_cls shape: (2444, 1)
      


In [74]:
X_train_reg = pd.read_csv(os.path.join(splits_path, "X_train_reg.csv"))
y_train_reg = pd.read_csv(os.path.join(splits_path, "y_train_reg.csv"))
X_test_reg = pd.read_csv(os.path.join(splits_path, "X_test_reg.csv"))
y_test_reg = pd.read_csv(os.path.join(splits_path, "y_test_reg.csv"))
X_val_reg = pd.read_csv(os.path.join(splits_path, "X_val_reg.csv"))
y_val_reg = pd.read_csv(os.path.join(splits_path, "y_val_reg.csv"))

print("""
      X_train_reg shape: {}, y_train_reg shape: {}
      X_val_reg shape: {}, y_val_reg shape: {}
      X_test_reg shape: {}, y_test_reg shape: {}
      """.format(X_train_reg.shape, y_train_reg.shape, X_val_reg.shape, y_val_reg.shape, X_test_reg.shape, y_test_reg.shape))


      X_train_reg shape: (25045, 11), y_train_reg shape: (25045, 1)
      X_val_reg shape: (2210, 11), y_val_reg shape: (2210, 1)
      X_test_reg shape: (2210, 11), y_test_reg shape: (2210, 1)
      
